In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

In [ ]:
from hotelling.spatial.census import build_grid_polygons

# CRS note: berlin.geojson is saved in EPSG:3035 (non-standard) — use load_boundary()
zensus_3035_polygons = build_grid_polygons(zensus_filtered, cell_size=100.0, boundary=berlin)
print(f"Cells after boundary clip: {len(zensus_3035_polygons)}")

# Fix CRS for boundary and lor files — ensure EPSG:3035 consistency
lor_3035 = lor.copy()
if lor_3035.crs is None or str(lor_3035.crs) != 'EPSG:3035':
    if str(lor_3035.crs) == 'EPSG:4326':
        lor_3035 = lor_3035.set_crs('EPSG:3035', allow_override=True)
    else:
        lor_3035 = lor_3035.to_crs('EPSG:3035')

boundary_3035 = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')
if boundary_3035.crs is None or str(boundary_3035.crs) != 'EPSG:3035':
    boundary_3035 = boundary_3035.set_crs('EPSG:3035', allow_override=True)

berlin_3035 = berlin.copy()
if berlin_3035.crs is None or str(berlin_3035.crs) != 'EPSG:3035':
    berlin_3035 = berlin_3035.set_crs('EPSG:3035', allow_override=True)


In [ ]:
from hotelling.spatial.admin import refine_shapes_selection, select_ringbahn_lor

# select_ringbahn_lor is the canonical pipeline wrapper.
# Here we call refine_shapes_selection directly to access all scored columns
# (population_density, distance_to_boundary, etc.) for visualization.
buffer_distance = 1800.0  # metres — larger than pipeline default for viz purposes
lor_ringbahn = refine_shapes_selection(
    shapes=lor_3035,
    boundary=boundary_3035.geometry.unary_union,
    population_grid=zensus_3035_polygons,
    buffer_distance=buffer_distance,
    extend_selection_by=11
)
import pandas as pd
pd.DataFrame(lor_ringbahn)


In [ ]:
import matplotlib.pyplot as plt

lor_ringbahn_select = lor_ringbahn[lor_ringbahn['selected']].copy()


if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    lor_3035.plot(ax=ax, edgecolor='gray', facecolor='none', linewidth=0.5)
    lor_ringbahn_select.plot(ax=ax, column = "population_density", edgecolor='black', cmap='viridis', legend=True)
    boundary_3035.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2)
    ax.set_title(f'LOR Districts within {buffer_distance*10**(-3)}km of Ringbahn Boundary')
    plt.show()


In [ ]:
# HERE: OPTIMAL RECTANGLE
from hotelling.spatial.admin import find_optimal_rectangle

# NOTE: These parameters match the defaults in run_default_data_pipeline()
# (rect_buffer_distance=350.0, rect_augment_layers=(2, 0, 4, 2))
# Any change here should be reflected in exe.py as well.
# ── Parameters ────────────────────────────────────────────────────────────────
# buffer_distance: expand the Ringbahn boundary before fitting the rectangle.
# Set to 0.0 if no buffer is desired; increase to guarantee extra margin around
# the boundary (e.g. 500 m to keep the outer Ringbahn rail inside).
RECT_BUFFER_DISTANCE  = 350.0        # metres
CELL_SIZE             = 100.0        # metres — must match the Zensus grid step
MAX_ITERATIONS        = 10_000       # search space: sqrt(10000)=100 extra cols/rows
TOLERANCE             = 0.01         # 1 % relative density improvement to prefer larger rect

# augument_rectangle_by_additional_layers = [top, right, bottom, left]
# Each value is a number of extra 100 m grid-cell layers added to the respective
# side of the optimal rectangle AFTER the optimisation.
AUGMENT_LAYERS = [2, 0, 4, 2]

# ── Find optimal rectangle ────────────────────────────────────────────────────
# boundary  : the Ringbahn polygon (GeoSeries, EPSG:3035)
# population_grid : Zensus 100 m point grid — must have an 'Einwohner' column
optimal_rect = find_optimal_rectangle(
    boundary=boundary_3035.geometry,
    population_grid=zensus_filtered,
    buffer_distance=RECT_BUFFER_DISTANCE,
    cell_size=CELL_SIZE,
    augument_rectangle_by_additional_layers=AUGMENT_LAYERS,
    max_iterations=MAX_ITERATIONS,
    tolerance=TOLERANCE,
)

# ── Summary ───────────────────────────────────────────────────────────────────
print("Optimal rectangle summary")
print("─" * 50)
print(f"  Grid dimensions : {optimal_rect['n_cols'].iloc[0]} cols × {optimal_rect['n_rows'].iloc[0]} rows")
print(f"  Width           : {optimal_rect['width_m'].iloc[0]:,.0f} m")
print(f"  Height          : {optimal_rect['height_m'].iloc[0]:,.0f} m")
print(f"  Centre          : ({optimal_rect['center_x'].iloc[0]:.1f}, {optimal_rect['center_y'].iloc[0]:.1f})  [EPSG:3035]")
print(f"  Population      : {optimal_rect['population'].iloc[0]:,}")
print(f"  Pop. density    : {optimal_rect['population_density'].iloc[0]:.4e}  residents / m²")
minx, miny, maxx, maxy = optimal_rect.geometry.iloc[0].bounds
print(f"  Bounds          : minx={minx:.1f}, miny={miny:.1f}, maxx={maxx:.1f}, maxy={maxy:.1f}")

# ── Visualisation ─────────────────────────────────────────────────────────────
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    lor_3035.plot(ax=ax, edgecolor="gray", facecolor="none", linewidth=0.5, label="All LOR districts")
    lor_ringbahn.plot(
        ax=ax, column="population_density", cmap="Blues",
        edgecolor="steelblue", alpha=0.5, linewidth=0.6, legend=False,
        label="Ringbahn LOR (selected)",
    )
    boundary_3035.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=2.0, label="Ringbahn boundary")
    optimal_rect.plot(ax=ax, edgecolor="lime", facecolor="none", linewidth=2.5, label="Optimal rectangle")
    ax.set_title("Optimal Population-Dense Rectangle Enclosing the Ringbahn Boundary", fontsize=13)
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
from hotelling.spatial.census import build_full_grid

full = build_full_grid(boundary=optimal_rect, zensus=zensus)
full.to_parquet(PATH_PROCESSED / 'pop_grid.parquet', index=False)
print(f"pop_grid saved: {len(full)} cells")


In [ ]:
import contextily as ctx

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=300)
    full.plot(ax=ax, column='Einwohner', cmap='viridis', legend=True, figsize=(20, 10), markersize=1, alpha = 0.2, marker ='s')
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=2)
    lor.plot(ax=ax, edgecolor='gray', facecolor='none', linewidth=0.5)
    boundary_3035.plot(ax=ax, edgecolor='firebrick', facecolor='none', linewidth=0.6, label="Ringbahn LOR")
    ctx.add_basemap(ax, crs=full.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.5)
    ax.set_axis_off()
    plt.show()


In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_01_lor.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")